In [14]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

# ==================================================
# CAMINHO DO JSON
# ==================================================
json_path = r"plots/laser1064/all_pipelines_summary.json"

# ==================================================
# LEITURA
# ==================================================
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# ==================================================
# EXTRAÇÃO DOS DADOS
# ==================================================
table_data = {}

for pipeline_info in data["pipelines"]:

    pipeline_name = pipeline_info["pipeline"]

    mean_acc = pipeline_info["mean_accuracy"] * 100
    std_acc = pipeline_info["std_accuracy"] * 100

    # Ex:
    # none_svm_linear
    # snv_random_forest
    # savgol_snv_pls_da

    if pipeline_name.startswith("none_"):
        preprocess = "None"
        model = pipeline_name.replace("none_", "")

    elif pipeline_name.startswith("snv_"):
        preprocess = "SNV"
        model = pipeline_name.replace("snv_", "")

    elif pipeline_name.startswith("savgol_snv_"):
        preprocess = "SG + SNV"
        model = pipeline_name.replace("savgol_snv_", "")

    else:
        continue

    if preprocess not in table_data:
        table_data[preprocess] = {}

    table_data[preprocess][model] = (
        f"{mean_acc:.2f} ± {std_acc:.2f}"
    )

# ==================================================
# DATAFRAME
# ==================================================
df = pd.DataFrame.from_dict(table_data, orient="index")

# ordem desejada
row_order = ["None", "SNV", "SG + SNV"]
col_order = ["svm_linear", "random_forest", "pls_da"]

df = df.reindex(row_order)
df = df[col_order]

# nomes bonitos
df.columns = [
    "SVM Linear",
    "Random Forest",
    "PLS-DA"
]

df.index.name = "Pré-processamento"

# ==================================================
# PLOT DA TABELA
# ==================================================
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axis("off")

table = ax.table(
    cellText=df.values,
    rowLabels=df.index,
    colLabels=df.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2)

plt.tight_layout()

# ==================================================
# SALVAMENTO
# ==================================================
output_dir = os.path.dirname(json_path)

output_path = os.path.join(
    output_dir,
    "tabela_resultados.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print(f"Tabela salva em:\n{output_path}")

Tabela salva em:
plots/laser1064\tabela_resultados.png
